# Silver: governed customer

**Audience:** data engineers validating medallion architecture and AIDP lineage.

**Prerequisites:** the canonical lab assets, shared compute and five job parameters.

**Learning goals:** trace governed transformations, verify isolation, and inspect deterministic results.


In [ ]:
import re
from functools import reduce
from pyspark.sql import Window, functions as F

# oidlUtils is injected by AIDP Workbench; no import is required.
def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name, "")
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")

participant_match = re.fullmatch(r"u([1-9][0-9]*)", participant_key)
if participant_match is None or int(participant_match.group(1)) < 101:
    raise ValueError("Invalid participant_key")
if lab_id != "telco_lineage":
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")

layer_prefixes = {"landing": "01_landing", "bronze": "02_bronze", "silver": "03_silver", "gold": "04_gold"}

def table(layer, logical_name):
    return f"aidp_lab.oci_{layer}.{participant_key}_{lab_id}_{logical_name}"

def location(layer, logical_name):
    return f"oci://{bucket_name}@{objectstorage_namespace}/{layer_prefixes[layer]}/users/{participant_key}/{lab_id}/{logical_name}/"

def write_delta(frame, layer, logical_name, _ddl):
    target = table(layer, logical_name)
    target_location = location(layer, logical_name)
    (frame.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").option("path", target_location)
        .saveAsTable(target))
    actual = spark.table(target).count()
    assert actual == frame.count(), f"Delta count mismatch for {logical_name}"
    print(f"Delta {layer}.{logical_name}: {actual} rows")


## Transformation

Run this cell once. It is idempotent and checks its row-level contract.


In [ ]:
customers = spark.table(table("bronze", "crm_customers"))
customer_window = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc(), F.col("source_row_id").desc())
ranked_customers = customers.withColumn("_rank", F.row_number().over(customer_window))
customer_reason = (F.when(F.col("_rank") > 1, F.lit("duplicate_customer"))
    .when(~F.col("document_number").rlike("^[0-9]{8}$"), F.lit("invalid_document")))
ranked_customers = ranked_customers.withColumn("_reason", customer_reason)
customer_master = (ranked_customers.filter(F.col("_reason").isNull())
    .select("participant_key", "customer_id",
        F.concat_ws(" ", "first_name", "last_name").alias("full_name"),
        "document_number", F.lower("segment").alias("segment"),
        F.lower("status").alias("status"), F.to_timestamp("updated_at").alias("updated_at")))

customer_issues = (ranked_customers.filter(F.col("_reason").isNotNull())
    .select(F.lit(participant_key).alias("participant_key"), F.lit("crm_customers").alias("dataset"),
        "source_row_id", F.col("customer_id").alias("record_key"),
        F.col("_reason").alias("reason_code"), F.current_timestamp().alias("quarantined_at")))

addresses = spark.table(table("bronze", "crm_addresses"))
address_check = addresses.join(customer_master.select("customer_id").withColumn("_customer_ok", F.lit(True)), "customer_id", "left")
address_issues = (address_check.filter(F.col("_customer_ok").isNull())
    .select(F.lit(participant_key).alias("participant_key"), F.lit("crm_addresses").alias("dataset"),
        "source_row_id", F.col("address_id").alias("record_key"),
        F.lit("orphan_customer").alias("reason_code"), F.current_timestamp().alias("quarantined_at")))
customer_addresses = (address_check.filter(F.col("_customer_ok").isNotNull())
    .select("participant_key", "address_id", "customer_id", F.lower("address_type").alias("address_type"),
        "address_line", "city", "province", "region",
        F.col("is_primary").cast("boolean").alias("is_primary"),
        F.to_timestamp("updated_at").alias("updated_at")))

products = spark.table(table("bronze", "product_catalog"))
product_catalog = products.select(
    "participant_key", "product_id", F.upper("service_type").alias("service_type"),
    "product_name", "product_family", F.col("monthly_fee").cast("decimal(12,2)").alias("monthly_fee"),
    F.lower("status").alias("status"), F.to_timestamp("updated_at").alias("updated_at"))

write_delta(customer_master, "silver", "customer_master", "participant_key STRING, customer_id STRING, full_name STRING, document_number STRING, segment STRING, status STRING, updated_at TIMESTAMP")
write_delta(customer_addresses, "silver", "customer_addresses", "participant_key STRING, address_id STRING, customer_id STRING, address_type STRING, address_line STRING, city STRING, province STRING, region STRING, is_primary BOOLEAN, updated_at TIMESTAMP")
write_delta(product_catalog, "silver", "product_catalog", "participant_key STRING, product_id STRING, service_type STRING, product_name STRING, product_family STRING, monthly_fee DECIMAL(12,2), status STRING, updated_at TIMESTAMP")
customer_issues.unionByName(address_issues).write.format("delta").mode("overwrite").save(location("silver", "_quality/customer"))
assert customer_master.count() == 493 and customer_addresses.count() == 617 and product_catalog.count() == 15


## Exercise and common pitfall

**Exercise:** follow one customer or service identifier into the next task and explain every derived column.

**Answer scaffold:** identify the source table, join key, transformation and target column.

**Pitfall:** never replace the job parameters with participant-specific literals; doing so breaks canonical hashes and isolation.

**Extension:** inspect the resulting entity and column lineage in Master Catalog.
